# vime GRPO 训练链路详解

## 写给 RL / 分布式训练新手

这份 Notebook 会把 vime 中 GRPO（Group Relative Policy Optimization）的**完整训练链路**讲清楚——从 128 条 rollout 数据出发，穿过组归一化、优势计算、microbatch 切分、前向/反向传播，直到 optimizer.step()。

**你不会只看到代码 — 你会看到每一步的数据形状变化、在哪个文件发生、以及为什么这样设计。**

### 阅读指南

1. **术语速查** — 先了解关键词
2. **数据全貌** — 128 条数据从哪来、长什么样
3. **GRPO 组归一化** — reward 怎么变成 advantage（核心算法）
4. **数据切分** — 128 条怎么分配到 GPU / step / microbatch
5. **训练前准备** — old/ref log_probs 前向 + 优势广播
6. **Policy Loss** — PPO 截断目标 + loss 归一化
7. **训练循环** — forward、backward、opt.step 三件事
8. **与 verl 的对比** — 算法一致，工程取舍不同
9. **NPU 训练基础设施** — vllm-ascend / MindSpeed / Megatron 怎么连起来
10. **总结** — 关键约定 + 完整文件清单

## 1. 关键术语速查

| 缩写 | 全称 | 通俗解释 |
|------|------|---------|
| **GRPO** | Group Relative Policy Optimization | 组相对策略优化：同 prompt 的 N 个回答互相比较，reward 减组均值作为 advantage |
| **GAE** | Generalized Advantage Estimation | PPO 的经典 advantage 算法：用 value 网络 + 时序差分估计每步 advantage |
| **PPO** | Proximal Policy Optimization | 策略优化算法，用 clip 限制每次更新幅度，防止策略「跳太远」 |
| **rollout** | 数据采样 | 让模型「生成」N 条回答，送给 reward 模型打分，得到 (prompt, response, reward) 三元组 |
| **n_samples_per_prompt** | 每 prompt 采样数 | 同一个 prompt 生成多少条不同回答，默认 GRPO 用 16 |
| **microbatch** | 微批次 | 一次 GPU forward 处理的样本数；多个 microbatch 的梯度累积后做一次 opt.step |
| **DP** | Data Parallel | 数据并行：每个 GPU 持完整模型副本，不同 GPU 处理不同样本 |
| **TP** | Tensor Parallel | 张量并行：把一个大矩阵沿某维切开分到多个 GPU |
| **CP** | Context Parallel | 上下文并行：把长序列切成多段分到多个 GPU |
| **PP** | Pipeline Parallel | 流水线并行：把模型层分到不同 GPU，像流水线一样传递中间结果 |
| **gbs** | global batch size | gbs = 一次 opt.step 用的 rollout 总数 |
| **TIS** | Truncated Importance Sampling | 截断重要性采样：用 rollout 时的 log_probs 修正 off-policy 偏差 |
| **OPSM** | Off-Policy Sequence Masking | 离线策略序列掩码：检测并丢弃 advantage 为负且 KL 过大的过期样本 |
| **DAPO** | Decoupled Alignment Policy Optimization | 解耦对齐：PPO 变体，高低 clip 独立设置（如 eps_clip=0.2, eps_clip_high=0.28） |

## 2. 数据全貌：128 条数据从哪来、长什么样

### 2.1 一次 rollout 产出什么

```
rollout_batch_size  = 8     # 8 个不同的 prompt（题目）
n_samples_per_prompt = 16   # 每个 prompt 生成 16 条不同回答
─────────────────────────────────
总样本数 = 8 × 16 = 128 条
```

每个样本是一个 Python 对象（`vime/rollout/data_source.py` 的 `Sample`），包含：

| 字段 | 形状/类型 | 说明 |
|------|----------|------|
| `tokens` | `[prompt_len + response_len]` 的 int 列表 | 完整 token 序列 |
| `response_length` | int | 模型生成部分的 token 数 |
| `reward` | float | reward 模型给出的标量分数 |
| `loss_mask` | `[response_length]` 的 0/1 列表 | 哪些 response token 参与 loss 计算 |
| `rollout_log_probs` | `[response_length]` 的 float 列表 | vLLM 生成时记录的 log 概率（用于 TIS） |

### 2.2 「组」的概念

GRPO 的核心：**同一个 prompt 的 n_samples_per_prompt 条样本构成一个「组」**。组的 baseline = 组内样本 reward 的均值。

```
prompt 0: [sample_00, sample_01, ..., sample_015]  ← 组 0（16 条）
prompt 1: [sample_10, sample_11, ..., sample_115]  ← 组 1（16 条）
...
prompt 7: [sample_70, sample_71, ..., sample_715]  ← 组 7（16 条）
```

### 2.3 数据流经的文件

| 步骤 | 文件 | 做什么 |
|------|------|--------|
| (1) | vLLM rollout 引擎 | 生成 128 条回答，reward 模型打分 |
| (2) | `vime/ray/rollout.py:_post_process_rewards()` | **GRPO 组归一化** |
| (3) | `vime/ray/rollout.py:_convert_samples_to_train_data()` | 组装 `train_data` dict |
| (4) | `vime/utils/dp_schedule.py:build_dp_schedule()` | 切分到 DP rank / step / microbatch |
| (5) | `vime/backends/megatron_utils/data.py:get_data_iterator()` | Megatron 训练数据迭代器 |
| (6) | `vime/backends/megatron_utils/model.py:train()` | **训练循环** |

## 3. GRPO 组归一化：reward 怎么变成 advantage

**这是 GRPO 最核心的一步。** 它不是在训练时发生的——而是在 rollout 数据收集阶段就完成了。

代码位置：`vime/ray/rollout.py:625-650` → `_post_process_rewards()`

### 3.1 伪代码

```python
# 输入：128 个 reward 标量
raw_rewards = [r0, r1, ..., r127]          # 128 个 float

# Step 1: reshape 成 [组数=8, 组内样本数=16]
rewards = torch.tensor(raw_rewards).reshape(8, 16)
# tensor([[r00, r01, ..., r015],           ← prompt 0 的 16 条
#         [r10, r11, ..., r115],           ← prompt 1 的 16 条
#         ...
#         [r70, r71, ..., r715]])          ← prompt 7 的 16 条

# Step 2: 减组内均值（GRPO baseline）
mean = rewards.mean(dim=-1, keepdim=True)  # 每组 16 个 reward 的均值
rewards = rewards - mean                    # 每条减去所在组的均值

# Step 3: 除以组内标准差（grpo_std_normalization=True，默认开启）
std = rewards.std(dim=-1, keepdim=True)
rewards = rewards / (std + 1e-6)

# Step 4: flatten 回 128 个标量 → 这就是 advantage
advantages = rewards.flatten()             # [128] float
```

### 3.2 数据形状变化

| 步骤 | 操作 | 输入形状 | 输出形状 | 说明 |
|------|------|----------|----------|------|
| 1 | `reshape(8, 16)` | `[128]` | `[8, 16]` | 按 prompt 分组 |
| 2 | `- mean(dim=-1)` | `[8, 16]` | `[8, 16]` | 去中心化（减去组 baseline） |
| 3 | `/ std(dim=-1)` | `[8, 16]` | `[8, 16]` | 归一化（除以组内标准差） |
| 4 | `flatten()` | `[8, 16]` | `[128]` | 展平回 128 个标量 advantage |

最终结果：每条样本拥有一个「组相对优势」标量
- advantage > 0 → 比组内平均好 → 鼓励
- advantage < 0 → 比组内平均差 → 惩罚
- advantage ≈ 0 → 与平均持平 → 中性

### 3.3 为什么在 rollout 阶段算而不是训练时算？

vime 的设计哲学：**把能提前算的都提前算好**。优势只依赖于 rewards（rollout 阶段已有），不依赖模型参数。这样做的好处：
- rollout 数据和训练计算解耦，训练侧只关心「优势 + tokens」
- reward 后处理 hook 可以自由替换（`--custom-reward-post-process-function-path`），不影响训练代码
- 同一批数据可以被不同算法变体共享

## 4. 数据切分：128 条怎么分配到 GPU / step / microbatch

代码位置：`vime/utils/dp_schedule.py:66-191` → `build_dp_schedule()`

### 4.1 核心公式

```python
# vime/utils/arguments.py:1804
global_batch_size = rollout_batch_size * n_samples_per_prompt // num_steps_per_rollout
num_steps = num_rollouts // global_batch_size         # = opt.step 次数
```

带入 128 条场景：
```python
# 默认 num_steps_per_rollout = 1
gbs = 8 * 16 // 1 = 128   →  num_steps = 128 // 128 = 1   # 整个 128 条做 1 次 opt.step

# num_steps_per_rollout = 4
gbs = 8 * 16 // 4 = 32    →  num_steps = 128 // 32 = 4     # 128 条分 4 step，每次 32 条
```

### 4.2 build_dp_schedule 四个阶段

```python
def build_dp_schedule(args, train_parallel_config, total_lengths, ...):
    # 阶段 1: 按 rollout_id 归组（同一 rollout 的样本必须在同 step 内）
    rollout_id_to_samples = group_by(rollout_indices)

    # 阶段 2: 切成 num_steps 个 step
    num_steps = len(rollout_ids) // global_batch_size
    for step_i in range(num_steps):
        step_rollouts = rollout_ids[step_i * gbs : (step_i + 1) * gbs]

        # 阶段 3: 每个 step 内打包成 K 个 microbatch
        if use_dynamic_batch_size:
            step_mbs = first_fit_pack(step_lengths, max_tokens_per_gpu * cp_size)
        else:
            step_mbs = fixed_chunk(micro_batch_size)

        # 阶段 4: 分配 K 个 microbatch 到 dp_size 个 rank
        K = max_align(len(step_mbs), dp_size * mb_group)
        per_rank = K // dp_size
        rank_indices = strided_round_robin(step_mbs, dp_size)
```

### 4.3 默认配置下 128 条的分发（DP=2 为例）

```
128 样本，1 step，DP=2，micro_batch_size=1，static batch

Step 0: 128 microbatches（每个 1 条样本）
  rank 0: mb 0, 2, 4, ..., 126 (64 个 mb)
  rank 1: mb 1, 3, 5, ..., 127 (64 个 mb)

每个 rank 做 64 次 forward/backward，梯度累积
最后 all-reduce 跨 rank → 1 次 opt.step
```

### 4.4 关键约束

- 每个 DP rank 的 microbatch 数**必须相同**（PP 同步要求）
- microbatch 总数必须能被 `dp_size * (mb_group if vpp else 1)` 整除
- dynamic batch 下通过 `expand_bins_by_splitting` 自动对齐

## 5. 训练前准备：old/ref log_probs 前向 + 优势广播

代码位置：`vime/backends/megatron_utils/actor.py:440-519` → `train_actor()`

训练前做三件事：跑 ref 得到 KL 基准、跑 actor 得到 π_old、把 rollout 阶段的标量优势广播到每个 token。

### 5.1 train_actor 调度逻辑

```python
def train_actor(self, rollout_id, rollout_data, external_data=None):
    data_iterator = get_data_iterator(rollout_data)

    # Step 1: 如果有 ref 模型，前向得到 ref_log_probs（KL 基准）
    if "ref" in weights_backuper:
        switch_model("ref")
        rollout_data["ref_log_probs"] = forward_only(
            get_log_probs_and_entropy, model, data_iterator)

    # Step 2: 前向 actor 得到 old_log_probs（π_old）
    can_reuse = (                                           # 10 个条件
        len(num_microbatches) == 1                          # 单 step
        and args.loss_type == "policy_loss"
        and args.kl_coef == 0
        and not args.use_rollout_logprobs
        and not args.get_mismatch_metrics
        and not args.use_critic
        and not args.keep_old_actor
        and not args.use_opd
        and not args.use_routing_replay
        and args.advantage_estimator != "gspo"
    )
    if not can_reuse:
        switch_model("actor")
        rollout_data["log_probs"] = forward_only(...)

    # Step 3: 计算 advantages 和 returns（标量 → token 广播）
    switch_model("actor")
    compute_advantages_and_returns(args, rollout_data)   # → loss.py:574
```

### 5.2 GRPO 路径：标量优势广播到 token

```python
# loss.py:635-639
elif args.advantage_estimator in ["grpo", "gspo"]:
    rewards = torch.tensor(rewards, dtype=float32, device=kl[0].device)  # [128]
    returns = get_grpo_returns(rewards, kl)          # 广播到每个 token
    advantages = [r for r in returns]

# ppo_utils.py:201-208
def get_grpo_returns(rewards, kl):
    """ones_like(kl[i]) * rewards[i] — 每条样本的每个 token 复制相同的优势标量"""
    returns = []
    for i in range(len(rewards)):
        returns.append(torch.ones_like(kl[i]) * rewards[i])
    return returns
```

广播效果：
| 输入 | 形状 | 输出 | 形状 |
|------|------|------|------|
| `rewards[0]` | 标量（如 0.5） | `returns[0]` | `[resp_len_0]` 全是 0.5 |
| `rewards[1]` | 标量（如 -0.3） | `returns[1]` | `[resp_len_1]` 全是 -0.3 |
| … | … | … | … |

即：rslot 阶段算好的组相对优势标量，被复制到每条回答的**每一个 token** 上，作为 PPO loss 中每个 token 的 advantage。

### 5.3 normalize_advantages — 二次归一化（可选）

```python
# loss.py:691 — 即使 rollout 已做组归一化，训练前还能再做跨 DP 全局 whiten
if args.normalize_advantages:
    all_advs = torch.cat(advantages)
    whitened = distributed_masked_whiten(
        all_advs, loss_masks, process_group=dp_group)
    advantages = split(whitened, chunk_lengths)
```

### 5.4 单步 on-policy 时 ratio≈1，但梯度有效

当 `can_reuse_log_probs_in_loss=True`（单 step、kl_coef=0）时：
- 跳过单独的 old_log_prob 前向
- 在 loss 中 `old_log_probs = log_probs.detach()` → `ratio = exp(0) = 1`
- PG loss 退化为：**∇(-1·A) = -A·∇log_probs**，即 REINFORCE 策略梯度

这是 GRPO 的设计行为，不是 bug。

## 6. Policy Loss：PPO 截断目标 + loss 归一化

代码位置：`vime/backends/megatron_utils/loss.py:800-1028` → `policy_loss_function()`

### 6.1 policy_loss_function 完整流程

```python
def policy_loss_function(args, batch, logits, sum_of_sample_mean):
    # Step 1: 从 logits 重算当前策略 log_probs（π_θ，带梯度）
    log_probs = get_log_probs_and_entropy(logits, ...)

    # Step 2: 取 old_log_probs（π_old，detach，无梯度）
    old_log_probs = batch["rollout_log_probs"] if use_rollout_logprobs
                    else batch.get("log_probs")

    # Step 3: 取 advantages（train_actor 阶段广播到 token 的）
    advantages = cat(batch["advantages"])

    # Step 4: 计算 PPO KL
    ppo_kl = old_log_probs - log_probs        # = log(π_old) - log(π_θ)
    # 单步 on-policy: old = log.detach() → ppo_kl ≡ 0

    # Step 5: PPO 截断目标（核心公式）
    pg_loss, pg_clipfrac = compute_policy_loss(
        ppo_kl, advantages, eps_clip, eps_clip_high)

    # Step 6: 聚合 loss
    loss = sum_of_sample_mean(pg_loss) - entropy_coef * entropy
    if use_kl_loss:
        loss += kl_loss_coef * KL(π_θ || π_ref)
    return loss, metrics
```

### 6.2 compute_policy_loss — PPO 截断

```python
# ppo_utils.py:125-148
def compute_policy_loss(ppo_kl, advantages, eps_clip, eps_clip_high):
    ratio = exp(-ppo_kl)                            # = π_θ / π_old
    pg_losses1 = -ratio * advantages                # 无 clip
    pg_losses2 = -clamp(ratio, 1-ε, 1+ε_high) * advantages  # 有 clip
    pg_loss = max(pg_losses1, pg_losses2)           # 取保守的

    # Dual-clip PPO (DAPO): 负 advantage 再加下界约束
    if eps_clip_c is not None:
        pg_losses3 = -eps_clip_c * advantages
        pg_loss = where(advantages < 0,
                        min(pg_losses3, pg_loss),
                        pg_loss)
    return pg_loss, clipfrac
```

### 6.3 sum_of_sample_mean — 默认 loss 归一化

```python
# cp_utils.py:53 — 默认行为（calculate_per_token_loss=False）
def sum_of_sample_mean(x):
    """每条样本: token 加权平均。样本之间: 等权。"""
    return sum([
        (x_i * loss_mask_i).sum() / clamp(denom, 1)
        for x_i, loss_mask_i, denom in zip(samples, masks, denoms)
    ])
```

效果：长序列和短序列对梯度的贡献相同 → 与 verl 默认的 `token-mean`（长序列权重更大）不同。

### 6.4 Megatron 集成：loss 缩放

```python
# loss.py:1224
if not args.calculate_per_token_loss:
    loss = loss * num_microbatches \
           / step_global_batch_size \
           * mpu.get_data_parallel_world_size(with_context_parallel=True)
```

目的：抵消 Megatron 内部的梯度累积缩放，让最终梯度恰好等于「对全局 128 条样本求平均」。

## 7. 训练循环：forward、backward、opt.step 三件事

代码位置：`vime/backends/megatron_utils/model.py:310-498` → `train_one_step()`

### 7.1 train_one_step 完整伪代码

```python
def train_one_step(data_iterator, model, optimizer,
                   num_microbatches, step_global_batch_size):
    # ====== (1) 清梯度 ======
    for model_chunk in model:
        model_chunk.zero_grad_buffer()
    optimizer.zero_grad()

    # ====== (2) forward + backward ======
    # Megatron 流水线引擎：逐 microbatch forward → loss_function → backward
    # 梯度自动累积到 optimizer 的 grad buffer
    losses_reduced = forward_backward_func(
        forward_step_func = forward_step,     # model(tokens) → logits + loss_fn
        data_iterator     = data_iterator,
        num_microbatches  = num_microbatches, # per-rank 的 mb 数
        forward_only      = False,            # ← backward 会执行
    )

    # ====== (3) grad 安全检查 ======
    found_inf = optimizer.prepare_grads()       # all-reduce 梯度
    valid_step = not found_inf
    if valid_step:
        grad_norm = optimizer.get_grad_norm()
        valid_step = not (isnan(grad_norm) or isinf(grad_norm))

    # ====== (4) optimizer.step ======
    if valid_step:
        update_successful, grad_norm, _ = optimizer.step()
        opt_param_scheduler.step(increment=step_global_batch_size)

    # ====== (5) 再次清梯度 ======
    for model_chunk in model:
        model_chunk.zero_grad_buffer()
    optimizer.zero_grad()
```

### 7.2 forward_step — 每个 microbatch 内部

```python
def forward_step(data_iterator, model):
    # 1) 取这个 microbatch 的数据
    batch = get_batch(data_iterator, [
        "tokens", "loss_masks", "log_probs",
        "advantages", "response_lengths", ...])
    # 2) 前向传播（logits shape = [1, T, vocab_size]）
    logits = model(input_ids=batch["tokens"], ...)
    # 3) 返回 logits + 部分应用的 loss_function
    return logits, partial(loss_function, args, batch, num_microbatches)
```

### 7.3 完整数据流四阶段

| 阶段 | 入口文件 | 核心逻辑 |
|------|---------|---------|
| **Phase A: Rollout 处理** | `vime/ray/rollout.py` | 128 条 → `_post_process_rewards()` 组归一化 → `advantages[128]` 标量 → `_convert_samples_to_train_data()` → `train_data` dict |
| **Phase B: 调度切分** | `vime/utils/dp_schedule.py` | 128 条 → step 0 (gbs=128) → K 个 microbatch → 按 dp_size 分配给各 rank |
| **Phase C: 训练前准备** | `vime/backends/megatron_utils/actor.py` | ref forward → `ref_log_probs` (if kl>0) → actor forward → `old_log_probs` (π_old) → `compute_advantages_and_returns()` 广播到 token |
| **Phase D: Megatron 训练** | `vime/backends/megatron_utils/model.py` | 逐 mb: `model.forward(tokens)` → `policy_loss_function(logits, A, old_lp)` → `compute_policy_loss(ppo_kl, A, ε)` → `sum_of_sample_mean(pg_loss)` → `loss.backward()`; 所有 mb 累积梯度 all-reduce → `optimizer.step()` → `opt_param_scheduler.step()` |

### 7.4 一次 opt.step 用多少数据？

| 参数 | 默认值 | 含义 |
|------|--------|------|
| `num_steps_per_rollout` | 1 | 整个 rollout 分几个 step |
| `global_batch_size` | 128 | 每 step 的 rollout 数 |
| `num_steps` (= opt.step 次数) | 1 | 128 条一次更新 |
| per-rank microbatch | 64(dp=2) | 每个 rank 的梯度累积步数 |

**结论：默认配置下，128 条样本的全部梯度累积为 1 次 opt.step。**

## 8. 与 verl 的对比

[verl](https://github.com/volcengine/verl)（`/home/g00841271/verl`）是另一个广泛使用的 RLHF 训练框架。算法内核完全相同（组归一化 + PPO 截断），区别在工程取舍。

### 8.1 GRPO 归一化

| | vime | verl |
|---|---|---|
| 计算位置 | **Rollout 侧** `_post_process_rewards` | **Driver 侧** `compute_advantage` |
| 分组方式 | `reshape(-1, n_samples_per_prompt)` 靠顺序 | `index/uid` 显式 `defaultdict(list)` |
| 单样本组 | `n_samples_per_prompt==1` 时关 std | `len==1` → mean=0/std=1 |

### 8.2 数据复用 — 最大区别

**vime**：一次 rollout 的样本**只用一遍**。用 `num_steps_per_rollout` 把 128 条互斥切成 N 个 step。

**verl**：有 `ppo_epochs` 参数，同一批数据可以**重复训练多遍**：
```python
# verl dp_actor.py:558
for _ in range(ppo_epochs):                  # ← 数据复用
    for mini_batch in mini_batches:          # ← 切 mini-batch
        for mb in mini_batch.split(...):
            loss.backward()                  # ← 梯度累积
        optimizer_step()                     # ← 每个 mini-batch 一次
```

`ppo_epochs > 1` 时，同一条样本被多次用于梯度更新（真正的 off-policy PPO，ratio 显著偏离 1）。

> vime 走 off-policy 用的是另一套：`keep_old_actor` + `update_weights_interval`。

### 8.3 其他差异汇总

| | vime | verl (FSDP) |
|---|---|---|
| 梯度累积 | Megatron `forward_backward_func` 引擎 | Python 手写 `for mb: loss.backward()` |
| 跨 DP 同步 | DDP + `all_reduce` | FSDP reduce-scatter |
| 后端 | **仅 Megatron** | **FSDP + Megatron 双后端** |
| loss 归一化默认 | per-rollout 等权 | **token-mean（长序列权重更大）** |
| Dr.GRPO | `--grpo-std-normalization False` | `norm_adv_by_std_in_grpo=False` |

### 8.4 对齐实验

| 目标 | vime | verl |
|------|------|------|
| 相同 loss 归一化 | `--calculate-per-token-loss` | 默认 token-mean |
| 相同数据复用 | `--num-steps-per-rollout 1` | `ppo_epochs=1` |

## 9. NPU 训练基础设施：vllm-ascend / MindSpeed / Megatron 的协作

本节覆盖 vime 对华为 Ascend NPU 的全栈适配，包含提交 `8d6c1841` 的 GDN port + sleep mode 改动以及所有未提交工作区改动。

### 9.1 三大外部依赖

| 依赖 | 路径 | 角色 |
|------|------|------|
| **vllm-ascend** | `/home/g00841271/vllm-ascend` | vLLM NPU 后端：HCCL 通信、CANN 算子、CaMemAllocator sleep 管理 |
| **MindSpeed** | `/home/g00841271/MindSpeed` | 昇腾训练加速库：NPU 适配器（`megatron_adaptor`）、GDN AscendC 算子、Triton causal_conv1d |
| **Megatron-LM** | `/home/g00841271/Megatron-LM` | 训练引擎：PP/DP/TP/CP 分布式并行 |

### 9.2 NPU 适配器：必须延迟加载

```python
# vime/backends/megatron_utils/__init__.py（未提交）
def _ensure_npu_adaptor():
    """延迟导入 mindspeed.megatron_adaptor。
    模块级 import 会导致 vLLM 子进程也拉入 mindspeed——
    mindspeed 的 torch.compile monkey-patch 会破坏 cudagraph 的 aot_compile 路径。
    """
    if is_npu():
        import mindspeed.megatron_adaptor
```

同样的保护还覆盖了：
- `update_weight_from_tensor.py` → `_device_module()`（§9.5）
- `mbridge/__init__.py` → `__getattr__` 延迟导入（§9.6）

### 9.3 vLLM 子进程环境清理

```python
# vime/backends/vllm_utils/vllm_engine.py（未提交）
# Docker 镜像为训练设置了 expandable_segments:True，
# 但 vllm-ascend 的 CaMemAllocator 在 sleep 模式下断言拒绝该参数
env.pop("PYTORCH_NPU_ALLOC_CONF", None)
# 父进程可能禁用了 torch.compile，但 vLLM 必须用它做 cudagraph capture
env.pop("TORCHDYNAMO_DISABLE", None)
```

### 9.4 Ray 环境变量透传

```python
# vime/ray/actor_group.py（未提交）
# Ray actors 不继承父进程环境
for _ev in ("VIME_SAVE_TIS_LOGPROBS",):
    if _ev in os.environ:
        env_vars[_ev] = os.environ[_ev]
```

### 9.5 权重同步：NPU/CUDA 双栈自动检测

```python
# vime/backends/megatron_utils/update_weight/update_weight_from_tensor.py（未提交）
def _device_module():
    """零 import 的 NPU/CUDA 检测。始终不拉入 mindspeed。"""
    try:
        import torch_npu
        return torch.npu
    except ImportError:
        return torch.cuda

# 所有 torch.cuda.* 替换为 _device_module().*:
_device_module().current_device()
_device_module().get_device_properties(idx).uuid
_device_module().ipc_collect()
```

### 9.6 mbridge 延迟导入

```python
# vime_plugins/mbridge/__init__.py（未提交）
# 从 21 行 eager import 改为 48 行 __getattr__ 延迟导入
# 原因: qwen2_5_vl 等子模块包含 mutable dataclass default，
# 在 vllm compilation config 未初始化时 import 会崩溃
def __getattr__(name: str):
    if name == "Qwen3_5Bridge":
        from .qwen3_5 import Qwen3_5Bridge as _cls
        return _cls
    ...
```

### 9.7 NPU inductor fix

```python
# vime/backends/megatron_utils/npu_inductor_fix.py（新增，untracked）
"""Monkey-patch torch._inductor.get_gpu_type()。
该函数 assert len(avail_gpus) <= 1，在 NPU device enumeration 下断言失败。
patch 后在 NPU 上返回 torch.npu 而不是崩溃。
"""
```

### 9.8 TIS 训练-推理一致性诊断

**Step 1** — 训练时保存 logprobs（`loss.py`，未提交）：
```python
if os.environ.get("VIME_SAVE_TIS_LOGPROBS"):
    torch.save({"old_log_probs": ..., "rollout_log_probs": ...}, save_path)
```

**Step 2** — 离线分析（`scripts/analyze_consistency.py`，新增，untracked）：
- 计算 abs_diff 的 7 个分位数（1%/5%/25%/50%/75%/95%/99%）
- Pearson / Spearman 相关系数 + Cosine similarity
- 输出 scatter data 用于可视化

用途：验证 Megatron 训练 logprobs 与 vLLM rollout logprobs 是否一致。不一致通常意味着权重同步 bug 或数值精度问题（如 §5.4 中 bf16→f32 cast 所修复的问题）。

### 9.9 NPU 上 bf16 logits 的 f32 cast

```python
# loss.py get_responses() + get_log_probs_and_entropy()（未提交）
# 原 assert logits.dtype == float32 → 改为条件 cast
# NPU mixed-precision 下 logits 可能是 bf16，需要 cast f32 做 log-softmax
if logits.dtype != torch.float32:
    logits = logits.float()
```

## 10. 总结

### 关键约定（改动必须保持一致性！）

1. **GRPO 优势是 rollout 阶段算的**：`_post_process_rewards` 在训练前就把 reward 变成组归一化标量。训练时只广播不重算。
2. **单步 on-policy 时 `ratio≡1`**：设计行为——`old_log_probs = log_probs.detach()` → ratio=1 → 退化为 REINFORCE 策略梯度。
3. **默认 per-rollout 等权**（非 token-mean）：loss 用 `sum_of_sample_mean`，每条 rollout 内 token 加权、rollout 间等权。要 token-mean 需 `--calculate-per-token-loss`。
4. **单次 rollout 内不复用数据**：用 `num_steps_per_rollout` 控制步数而非 `ppo_epochs`。
5. **NPU adaptor 必须延迟加载**：mindspeed 不能在 vLLM 子进程中 import（破坏 cudagraph）。所有相关文件都用 try/except + 延迟 import。

### 完整文件清单

| # | 文件 | 主题 | 状态 |
|---|------|------|------|
| 1 | `vime/ray/rollout.py` | GRPO 组归一化、训练数据组装 | 已提交 |
| 2 | `vime/utils/dp_schedule.py` | DP rank / step / microbatch 切分 | 已提交 |
| 3 | `vime/backends/megatron_utils/actor.py` | train_actor 调度 | 已提交 |
| 4 | `vime/backends/megatron_utils/loss.py` | advantage 广播 + policy_loss + TIS hook + bf16 cast | 已提交 + **未提交** |
| 5 | `vime/backends/megatron_utils/model.py` | forward/backward/opt.step | 已提交 |
| 6 | `vime/utils/ppo_utils.py` | PPO clip、GRPO returns 广播、KL | 已提交 |
| 7 | `vime/backends/megatron_utils/__init__.py` | NPU lazy adaptor | **未提交** |
| 8 | `vime/.../update_weight_from_tensor.py` | _device_module() NPU/CUDA 双栈 | **未提交** |
| 9 | `vime/backends/vllm_utils/vllm_engine.py` | vLLM 子进程环境清理 | **未提交** |
| 10 | `vime/ray/actor_group.py` | Ray env 透传 | **未提交** |
| 11 | `vime_plugins/mbridge/__init__.py` | mbridge 延迟导入 | **未提交** |
| 12 | `vime/backends/megatron_utils/npu_inductor_fix.py` | NPU inductor monkey-patch | **新增 untracked** |
| 13 | `scripts/analyze_consistency.py` | TIS 一致性分析 | **新增 untracked** |
| 14 | `scripts/run_qwen36_35b_a3b_dapo_math_npu.sh` | DAPO NPU 运行脚本 | **未提交改动** |

---

*最后更新: 2026-06-24*